**Experiment 1 — Synthetic proxy-discrimination demonstration.** Run top to bottom. All parameters, metrics, and I/O paths are contained in the single code cell below.

In [1]:
# 01_synthetic_proxy_discrimination.ipynb
#
# Experiment 1 — Synthetic demonstration of proxy discrimination and its mitigation
#
# Purpose:
#   Demonstrate the central claim of the AI-Gov-Alt framework using fully synthetic
#   data (no real personal information): removing a protected attribute alone does
#   NOT remove disparate impact, because correlated proxy variables retain the
#   group signal. The governance fairness engine (group-wise threshold calibration)
#   restores fairness while preserving predictive utility.
#
# Design:
#   - Protected attribute: nationality group (binary).
#   - Proxy variables: residential_region, spending_pattern (strongly group-correlated).
#   - Legitimate risk features: income, debt_ratio, pay_history.
#   - Historical label carries proxy-driven bias (label bias), mimicking biased past decisions.
#
# Feature-set conditions:
#   (1) with_protected        : all features including the protected attribute
#   (2) drop_protected_only   : protected attribute removed, proxies kept
#   (3) drop_proxies_too      : protected attribute AND proxies removed
#   (4) gov_engine            : proxies kept + group-wise threshold calibration (fairness engine)
#
# Metrics:
#   - AUC (predictive utility)
#   - Disparate Impact Ratio (DIR)
#   - Equalized-odds gaps (dTPR, dFPR)
#
# Outputs (relative to project root):
#   - results/tables/exp1_metrics.csv
#   - results/figures/exp1_auc_dir.(png|pdf)
#   - results/figures/exp1_dir_by_condition.(png|pdf)
#
# Visualization spec: seaborn, grayscale, no captions, dpi=600, saved as PNG and PDF.

# ==== Imports and global configuration ====
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Resolve project paths relative to the notebook location (notebooks/ -> project root)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()  # fallback if run from project root
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
FIG_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
for d in (DATA_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)

# Grayscale seaborn styling; figures saved without captions at dpi=600 as PNG and PDF
sns.set_theme(style="whitegrid")
sns.set_palette(sns.color_palette(["#000000", "#555555", "#999999", "#cccccc"]))
plt.rcParams.update({
    "figure.dpi": 600, "savefig.dpi": 600,
    "font.size": 11, "axes.edgecolor": "black", "axes.linewidth": 0.8,
    "grid.color": "0.85",
})

def save_fig(fig, name):
    # Save each figure as both PNG and PDF at dpi=600 with tight bounding box
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

# ==== Synthetic data generation ====
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
N = 8000

# Protected attribute: nationality group A (1) vs B (0)
protected = rng.binomial(1, 0.35, N)

# Proxy variables strongly correlated with the protected attribute
residential_region = 1.6 * protected + rng.normal(0, 0.6, N)
spending_pattern = 1.4 * protected + rng.normal(0, 0.6, N)

# Legitimate risk features, independent of the protected attribute
income = rng.normal(0, 1, N)
debt_ratio = rng.normal(0, 1, N)
pay_history = rng.normal(0, 1, N)

# Historical default label: legitimate risk PLUS proxy-driven (illegitimate) bias
true_risk = -0.9 * income + 0.8 * debt_ratio - 0.7 * pay_history
bias_term = 1.3 * residential_region + 1.1 * spending_pattern
logit = true_risk + bias_term + rng.normal(0, 0.5, N)
default = (logit > np.quantile(logit, 0.7)).astype(int)

df = pd.DataFrame({
    "protected": protected,
    "residential_region": residential_region,
    "spending_pattern": spending_pattern,
    "income": income,
    "debt_ratio": debt_ratio,
    "pay_history": pay_history,
    "default": default,
})
# Persist the generated dataset for reproducibility
df.to_csv(os.path.join(DATA_DIR, "synthetic_credit.csv"), index=False)

# ==== Fairness metric helpers ====
def disparate_impact_ratio(y_pred, group):
    # Approval is defined as predicted non-default (y_pred == 0)
    approve = (y_pred == 0).astype(int)
    r1 = approve[group == 1].mean()
    r0 = approve[group == 0].mean()
    return min(r1, r0) / max(r1, r0)

def equalized_odds_gaps(y_true, y_pred, group):
    def rate(cond_true, mask):
        idx = (y_true == cond_true) & mask
        return (y_pred[idx] == 1).mean() if idx.sum() else np.nan
    a, b = group == 1, group == 0
    d_tpr = abs(rate(1, a) - rate(1, b))
    d_fpr = abs(rate(0, a) - rate(0, b))
    return d_tpr, d_fpr

# ==== Experimental conditions ====
feature_sets = {
    "with_protected": ["protected", "residential_region", "spending_pattern",
                        "income", "debt_ratio", "pay_history"],
    "drop_protected_only": ["residential_region", "spending_pattern",
                            "income", "debt_ratio", "pay_history"],
    "drop_proxies_too": ["income", "debt_ratio", "pay_history"],
}

records = []
# Fixed split indices so all conditions are evaluated on the same test individuals
idx_train, idx_test = train_test_split(np.arange(N), test_size=0.30, random_state=1)
g_test = df["protected"].values[idx_test]
y_test = df["default"].values[idx_test]

for name, feats in feature_sets.items():
    X = df[feats].values
    model = LogisticRegression(max_iter=1000).fit(X[idx_train], df["default"].values[idx_train])
    proba = model.predict_proba(X[idx_test])[:, 1]
    y_pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y_test, proba)
    dir_v = disparate_impact_ratio(y_pred, g_test)
    d_tpr, d_fpr = equalized_odds_gaps(y_test, y_pred, g_test)
    records.append({"condition": name, "AUC": auc, "DIR": dir_v,
                    "dTPR": d_tpr, "dFPR": d_fpr})

# ==== Governance fairness engine: group-wise threshold calibration ====
# Keep proxies in the model but calibrate per-group thresholds to equalise approval rates.
feats = feature_sets["drop_protected_only"]
X = df[feats].values
model = LogisticRegression(max_iter=1000).fit(X[idx_train], df["default"].values[idx_train])
proba = model.predict_proba(X[idx_test])[:, 1]
auc = roc_auc_score(y_test, proba)
baseline_pred = (proba >= 0.5).astype(int)
target_approval = (baseline_pred == 0).mean()  # overall approval rate to match per group
th1 = np.quantile(proba[g_test == 1], target_approval)
th0 = np.quantile(proba[g_test == 0], target_approval)
gov_pred = np.where(g_test == 1, (proba >= th1).astype(int), (proba >= th0).astype(int))
dir_v = disparate_impact_ratio(gov_pred, g_test)
d_tpr, d_fpr = equalized_odds_gaps(y_test, gov_pred, g_test)
records.append({"condition": "gov_engine", "AUC": auc, "DIR": dir_v,
                "dTPR": d_tpr, "dFPR": d_fpr})

results = pd.DataFrame(records)
results.to_csv(os.path.join(TAB_DIR, "exp1_metrics.csv"), index=False)
print(results.round(3).to_string(index=False))

# ==== Figure 1: AUC vs DIR scatter across conditions ====
order = ["with_protected", "drop_protected_only", "drop_proxies_too", "gov_engine"]
plot_df = results.set_index("condition").loc[order].reset_index()
fig, ax = plt.subplots(figsize=(6, 4.2))
markers = ["o", "s", "^", "D"]
for i, row in plot_df.iterrows():
    ax.scatter(row["DIR"], row["AUC"], s=110, marker=markers[i],
               facecolor=["#000000", "#555555", "#999999", "#cccccc"][i],
               edgecolor="black", linewidth=1.0, label=row["condition"], zorder=3)
ax.axvline(0.8, color="black", linestyle="--", linewidth=0.9, zorder=1)
ax.set_xlabel("Disparate Impact Ratio (DIR)")
ax.set_ylabel("AUC")
ax.set_xlim(0.0, 1.05)
ax.legend(frameon=True, edgecolor="black", loc="lower left")
fig.tight_layout()
save_fig(fig, "exp1_auc_dir")
plt.close(fig)

# ==== Figure 2: DIR by condition bar chart ====
fig, ax = plt.subplots(figsize=(6, 4.2))
bars = sns.barplot(data=plot_df, x="condition", y="DIR", ax=ax,
                   edgecolor="black", linewidth=1.0)
for patch, shade in zip(ax.patches, ["#000000", "#555555", "#999999", "#cccccc"]):
    patch.set_facecolor(shade)
ax.axhline(0.8, color="black", linestyle="--", linewidth=0.9)
ax.set_xlabel("Condition")
ax.set_ylabel("Disparate Impact Ratio (DIR)")
ax.set_ylim(0, 1.05)
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
fig.tight_layout()
save_fig(fig, "exp1_dir_by_condition")
plt.close(fig)

print("Saved tables to:", TAB_DIR)
print("Saved figures to:", FIG_DIR)

          condition   AUC   DIR  dTPR  dFPR
     with_protected 0.992 0.299 0.265 0.103
drop_protected_only 0.992 0.303 0.250 0.089
   drop_proxies_too 0.716 1.000 0.485 0.073
         gov_engine 0.992 1.000 0.592 0.231


/tmp/ipykernel_534/2825097061.py:197: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")


Saved tables to: /home/claude/project/results/tables
Saved figures to: /home/claude/project/results/figures
